# Lesson 13: Experiments: break it on purpose

**Why:** the best way to understand why each part of the GPT exists is to **remove it and measure the damage**. Researchers call this an **ablation**.

All experiments use the **same small, fast model** (about 1.5 minutes each), the same random seed, and the same number of steps. The only difference is the one part we remove. Small models don't reach great scores, but we only care about the **differences** between versions.

| Experiment | What we remove | Lesson |
|---|---|---|
| baseline | nothing | |
| no positions | position embeddings: the model can't tell where characters are | 8 |
| no residual | the `x +` shortcuts in every block | 10 |
| no scaling | the ÷√hs in attention | 8 |
| no mask | the "no peeking" triangle: the model may read the future | 7 |
| no dropout | dropout, on a tiny dataset where memorizing is easy | 10 |

Running every cell takes **about 12–15 minutes** in total.

## 1. Setup: one helper that trains any model the same way

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from tinygpt.data import get_batch, load_text, split_data
from tinygpt.model import GPT, GPTConfig, Block, Head
from tinygpt.tokenizer import CharTokenizer

text = load_text()
tok = CharTokenizer.from_text(text)
train_data, val_data = split_data(tok.encode(text))

cfg = GPTConfig(vocab_size=tok.vocab_size, block_size=32, n_embd=64, n_head=4, n_layer=4, dropout=0.0)
STEPS = 800
results = {}

@torch.no_grad()
def evaluate(model, train, val, batches=40):
    model.eval()
    out = {name: sum(model(*get_batch(d, 32, cfg.block_size))[1].item() for _ in range(batches)) / batches
           for name, d in [("train", train), ("val", val)]}
    model.train()
    return out

def run(name, make_model, steps=STEPS, train=train_data, val=val_data, every=100):
    torch.manual_seed(1337)
    model = make_model()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
    curve, start = [], time.time()
    for step in range(steps + 1):
        if step % every == 0:
            curve.append((step, evaluate(model, train, val)))
        if step == steps:
            break
        xb, yb = get_batch(train, 32, cfg.block_size)
        _, loss = model(xb, yb)
        opt.zero_grad(); loss.backward(); opt.step()
    final = curve[-1][1]
    results[name] = {"curve": curve, "model": model, **final}
    print(f"{name:14}  train {final['train']:.3f}   val {final['val']:.3f}   ({time.time() - start:.0f}s)")
    return model

print(f"experiment model: {GPT(cfg).num_params():,} parameters, {STEPS} steps each")

## 2. The baseline: nothing removed

In [ ]:
baseline = run("baseline", lambda: GPT(cfg))

## 3. No position embeddings

We set every position vector to zero and freeze it, so the model gets **no information about order**.

**Predict:** does the model fall apart (loss back near the bigram's 2.4), get slightly worse, or barely notice? Hint: the mask still exists, and position 0 can only see 1 character while position 31 can see 32.

In [ ]:
def no_positions():
    model = GPT(cfg)
    nn.init.zeros_(model.pos_emb.weight)
    model.pos_emb.weight.requires_grad_(False)     # stays zero forever
    return model

run("no positions", no_positions);

## 4. No residual connections

We swap every block for a version without the `x +`. Each layer now **replaces** the information instead of adding to it (Lesson 10's wrap-up question).

**Predict:** better, worse, or about the same as the baseline?

In [ ]:
class NoResidualBlock(Block):
    def forward(self, x):
        x = self.attn(self.ln1(x))     # no "x +"
        x = self.ffwd(self.ln2(x))     # no "x +"
        return x

def no_residual():
    model = GPT(cfg)
    for block in model.blocks:
        block.__class__ = NoResidualBlock    # same parameters, new forward
    return model

run("no residual", no_residual);

## 5. No √hs scaling

We swap every attention head for a version that skips the `÷ √hs`. Here hs = 16, so the scores come out 4× too large, and softmax gets much "peakier" (Lesson 8).

**Predict:** better, worse, or about the same?

In [ ]:
class NoScaleHead(Head):
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.query(x), self.key(x), self.value(x)
        wei = q @ k.transpose(-2, -1)                                   # no "* hs ** -0.5"
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        self.last_wei = wei.detach()
        return self.dropout(wei) @ v

def no_scaling():
    model = GPT(cfg)
    for block in model.blocks:
        for head in block.attn.heads:
            head.__class__ = NoScaleHead
    return model

run("no scaling", no_scaling);

## 6. No mask: let it cheat

Every head may now see the **whole** chunk, including the characters it's supposed to predict.

**Predict:** what happens to the loss? Then, how good will its **writing** be?

In [ ]:
def no_mask():
    model = GPT(cfg)
    for block in model.blocks:
        for head in block.attn.heads:
            head.tril.fill_(1)       # every position may see every other position
    return model

cheater = run("no mask", no_mask)

start = torch.tensor([tok.encode("The alternating current ")])
torch.manual_seed(0)
print("\n--- baseline writes ---")
print(tok.decode(baseline.eval().generate(start, 200, temperature=0.8)[0].tolist()))
torch.manual_seed(0)
print("\n--- the cheater writes ---")
print(tok.decode(cheater.eval().generate(start, 200, temperature=0.8)[0].tolist()))

## 7. Overfitting, and how dropout helps

Our full Tesla text is big enough that a small model barely memorizes it. To see overfitting clearly, we train on just the **first 20,000 characters** (about 2% of the book) for longer, once **without** dropout and once **with** 30% dropout. Validation stays the usual 10% the model never sees.

**Predict:** without dropout, what happens to the **validation** loss as training goes on?

In [ ]:
tiny_train = train_data[:20_000]
SMALL_STEPS = 1500

run("tiny, no dropout", lambda: GPT(cfg), steps=SMALL_STEPS, train=tiny_train, every=250)
cfg_drop = GPTConfig(**{**cfg.__dict__, "dropout": 0.3})
run("tiny, dropout", lambda: GPT(cfg_drop), steps=SMALL_STEPS, train=tiny_train, every=250);

plt.figure(figsize=(9, 4))
for name, color in [("tiny, no dropout", "tab:red"), ("tiny, dropout", "tab:blue")]:
    steps = [s for s, _ in results[name]["curve"]]
    plt.plot(steps, [l["train"] for _, l in results[name]["curve"]], "--", color=color, label=f"{name}: train")
    plt.plot(steps, [l["val"] for _, l in results[name]["curve"]], "-o", color=color, label=f"{name}: validation")
plt.xlabel("step"); plt.ylabel("loss"); plt.ylim(0, 3.5); plt.legend(); plt.title("Memorizing 20,000 characters")
plt.show()

## 8. All results side by side

In [ ]:
names = ["baseline", "no positions", "no residual", "no scaling", "no mask"]
plt.figure(figsize=(9, 4))
for name in names:
    curve = results[name]["curve"]
    plt.plot([s for s, _ in curve], [l["val"] for _, l in curve], "-o", ms=3, label=name)
plt.xlabel("step"); plt.ylabel("validation loss"); plt.legend(); plt.title("Removing one part at a time")
plt.show()

print(f"{'experiment':18} {'train':>7} {'val':>7}   vs baseline (val)")
for name in names + ["tiny, no dropout", "tiny, dropout"]:
    r = results[name]
    diff = r["val"] - results["baseline"]["val"]
    print(f"{name:18} {r['train']:7.3f} {r['val']:7.3f}   {diff:+.3f}")

## 9. Final check

In [ ]:
assert results["no mask"]["train"] < results["baseline"]["train"], "cheating should look 'better' in training"
assert results["no residual"]["val"] > results["baseline"]["val"]
assert results["tiny, no dropout"]["val"] > results["tiny, dropout"]["val"], "dropout should help on tiny data"
print("Lesson 13 complete.")

## Wrap-up question

Which removed part hurt the **most**, and which hurt the **least**? Were you surprised by either?